# CS2309 — Precision trên macOS (MPS)

Bản **riêng cho Mac** — không sửa notebook Colab
[`CS2309_SwiftEdit_precision_disk_vram.ipynb`](./CS2309_SwiftEdit_precision_disk_vram.ipynb).

**Mục đích:** đo đóng góp **FP16 compute + EditCache** so với baseline FP32 trên Apple Silicon (MPS):
tốc độ, peak memory, PSNR vs FP32.

| Alias | Config | Ghi chú Mac |
|---|---|---|
| `fp32` | `baseline_fp32` | Reference |
| `fp16` | `improved_fp16_cache` | FP16 + EditCache (disk vẫn fp32) |
| `fp16_weight_xformers` | `fp16_disk_xformers` | **Không chạy trên MPS** (cần CUDA) |

**Batch nhỏ (Mac chậm):** 50 ảnh × 3 edit/ảnh = **150 job**.
Mỗi ảnh: edit 1 = cache miss, edit 2–3 = cache hit → thấy lợi EditCache.

**Attention trên MPS:** xFormers không có CUDA → pipeline dùng PyTorch
`scaled_dot_product_attention` (SDP). Không cần cài xformers.

**Ước lượng thời gian (M4, tham khảo):** fp32 ~15–30s/edit; fp16+cache ~5–8s/edit
→ cả 2 config ~1–2 giờ. Có thể hạ `MAX_JOBS` để smoke.

Output giống notebook gốc: `experimental_data/precision_run_*/` +
`bundle.zip` (quality / memory / disk / edited_images / run_meta).

### ⓪ Chọn config + batch

In [1]:
# Mac: có thể bật cả fp32 + fp16 trong một lần (eval chạy tuần tự).
# xFormers: để False trên MPS (sẽ bị SKIP nếu bật).
SELECT = {
    "fp32": True,
    "fp16": True,  # improved_fp16_cache
    "fp16_weight_xformers": False,  # cần CUDA — không dùng trên Mac
}

EVAL_CONFIGS = ",".join(k for k, on in SELECT.items() if on)
assert EVAL_CONFIGS, "Bật ít nhất 1 config"

# 50 ảnh × 3 edit = 150 job (edit 2–3 thường cache-hit)
N_IMAGES = 50
EDITS_PER_IMAGE = 3
MAX_JOBS = 150
SEED = 250101049

# True = chạy bash scripts/setup_macos.sh (lần đầu / thiếu deps).
# False = bỏ qua nếu .venv + weights đã sẵn.
RUN_SETUP = False

print("EVAL_CONFIGS =", EVAL_CONFIGS)
print(f"batch = {N_IMAGES} ảnh × {EDITS_PER_IMAGE} edit → MAX_JOBS={MAX_JOBS}")
print("SEED =", SEED)
print("RUN_SETUP =", RUN_SETUP)
if SELECT.get("fp16_weight_xformers"):
    print("NOTE: xFormers cần CUDA — trên MPS eval sẽ SKIP config này", flush=True)

EVAL_CONFIGS = fp32,fp16
batch = 50 ảnh × 3 edit → MAX_JOBS=150
SEED = 250101049
RUN_SETUP = False


### ① Kiểm tra macOS + MPS + PROJECT_ROOT

In [2]:
import os
import platform
import subprocess
import sys
from pathlib import Path


def _run_stream(cmd, *, cwd=None, env=None, check=True):
    """In log live (stdout+stderr)."""
    print("+", " ".join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    print(f"[exit {rc}]", flush=True)
    if check and rc != 0:
        raise RuntimeError(f"Command failed ({rc}): {cmd}")
    return rc


PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "SwiftEdit" / "infer.py").is_file():
    raise RuntimeError(
        f"Không thấy SwiftEdit/infer.py dưới {PROJECT_ROOT}. "
        "Mở notebook từ repo root hoặc thư mục notebooks/."
    )
os.chdir(PROJECT_ROOT)

# Ưu tiên .venv của repo
VENV_PY = PROJECT_ROOT / ".venv" / "bin" / "python"
PY = str(VENV_PY) if VENV_PY.is_file() else sys.executable

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

print("PROJECT_ROOT", PROJECT_ROOT, flush=True)
print("PY", PY, flush=True)
print("platform", platform.system(), platform.machine(), flush=True)

if platform.system() != "Darwin":
    print("WARNING: notebook này dành cho macOS; OS hiện tại:", platform.system(), flush=True)

import torch

print("torch", torch.__version__, flush=True)
print("mps available", torch.backends.mps.is_available(), flush=True)
print("cuda available", torch.cuda.is_available(), flush=True)
if not torch.backends.mps.is_available():
    raise RuntimeError("Cần PyTorch MPS (Apple Silicon). Kiểm tra .venv + requirements-mac.txt.")

# xFormers: xác nhận không có trên MPS (không fail)
try:
    import xformers  # noqa: F401

    print("xformers installed:", getattr(xformers, "__version__", "?"), "— vẫn cần CUDA để MEA", flush=True)
except ImportError:
    print("xformers: không cài (OK trên Mac — dùng SDP)", flush=True)

PROJECT_ROOT /Users/nguyenkz/Documents/code/CS2309.CH201
PY /Users/nguyenkz/Documents/code/CS2309.CH201/.venv/bin/python
platform Darwin arm64
torch 2.12.0
mps available True
cuda available False
xformers: không cài (OK trên Mac — dùng SDP)


### ② Setup macOS (tuỳ chọn) + weights

In [3]:
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

if RUN_SETUP:
    setup_sh = PROJECT_ROOT / "scripts" / "setup_macos.sh"
    _run_stream(["bash", str(setup_sh)], cwd=PROJECT_ROOT, env=env)
else:
    print("SKIP setup_macos.sh (RUN_SETUP=False)", flush=True)

# Metrics extras (PSNR/SSIM nếu có trong eval path)
_run_stream([PY, "-m", "pip", "install", "-q", "torchmetrics", "pyarrow"], cwd=PROJECT_ROOT)

WP32 = PROJECT_ROOT / "SwiftEdit" / "swiftedit_weights"
WP16 = PROJECT_ROOT / "SwiftEdit" / "swiftedit_weights_fp16"
fp32_ok = (WP32 / "sbv2_0.5").is_dir()
print("fp32 weights:", fp32_ok, WP32, flush=True)
print("fp16 weights (không bắt buộc cho fp16_cache):", (WP16 / "sbv2_0.5").is_dir(), flush=True)

if not fp32_ok:
    print("Thiếu Qualcomm fp32 — chạy setup_macos.sh (đặt RUN_SETUP=True) hoặc tải weights.", flush=True)
    raise FileNotFoundError(f"Thiếu {WP32}/sbv2_0.5")

# fp16_cache dùng disk fp32 + dtype fp16 lúc load — không cần convert disk.

SKIP setup_macos.sh (RUN_SETUP=False)
+ /Users/nguyenkz/Documents/code/CS2309.CH201/.venv/bin/python -m pip install -q torchmetrics pyarrow
[exit 0]
fp32 weights: True /Users/nguyenkz/Documents/code/CS2309.CH201/SwiftEdit/swiftedit_weights
fp16 weights (không bắt buộc cho fp16_cache): True


### ③ Dataset June17 (50 ảnh đầu × 3 prompt)

In [4]:
auto = PROJECT_ROOT / "data" / "PIE-Bench-auto200"
jobs_path = PROJECT_ROOT / "data" / "jobs_june17.json"

if not (auto / "annotation_images").is_dir():
    print("Freeze auto200...", flush=True)
    _run_stream([PY, "-u", str(PROJECT_ROOT / "scripts" / "freeze_piebench_auto200.py")], cwd=PROJECT_ROOT)

if not jobs_path.is_file():
    _run_stream([PY, "-u", str(PROJECT_ROOT / "scripts" / "build_june17_jobs.py")], cwd=PROJECT_ROOT)

import json

meta = json.loads(jobs_path.read_text(encoding="utf-8"))
print("jobs_june17:", jobs_path.is_file(), "n_jobs=", meta.get("n_jobs"), flush=True)
print(f"Eval sẽ lấy max {MAX_JOBS} job đầu (= {N_IMAGES} ảnh × {EDITS_PER_IMAGE} edit)", flush=True)

jobs_june17: True n_jobs= 600
Eval sẽ lấy max 150 job đầu (= 50 ảnh × 3 edit)


### ④ Eval → bundle.zip

Gọi cùng script với notebook Colab:
`scripts/run_precision_disk_vram_eval.py` →
`experimental_data/precision_run_<stamp>_<configs>/bundle.zip`.

In [5]:
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["OMP_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["OPENBLAS_NUM_THREADS"] = "1"

cmd = [
    PY,
    "-u",
    str(PROJECT_ROOT / "scripts" / "run_precision_disk_vram_eval.py"),
    "--configs",
    EVAL_CONFIGS,
    "--n-images",
    str(N_IMAGES),
    "--edits-per-image",
    str(EDITS_PER_IMAGE),
    "--jobs-manifest",
    str(PROJECT_ROOT / "data" / "jobs_june17.json"),
    "--weights-fp32",
    str(WP32),
    "--weights-fp16",
    str(WP16),
    "--seed",
    str(SEED),
    "--max-jobs",
    str(MAX_JOBS),
]
_run_stream(cmd, cwd=PROJECT_ROOT, env=env)

from precision_catalog import resolve_config_names

_canon = "+".join(resolve_config_names(EVAL_CONFIGS))
_exp = PROJECT_ROOT / "experimental_data"
_all = sorted(_exp.glob("precision_run_*/bundle.zip"), key=lambda p: p.stat().st_mtime)
_matched = [p for p in _all if _canon in p.parent.name]
BUNDLE_ZIP = (_matched or _all)[-1] if (_matched or _all) else None
print("canonical =", _canon, flush=True)
print("bundle candidates (newest last):", [str(p) for p in (_matched or _all)[-3:]], flush=True)
print("BUNDLE_ZIP =", BUNDLE_ZIP, flush=True)
if BUNDLE_ZIP is None:
    raise FileNotFoundError(
        f"Không thấy precision_run_*/bundle.zip trong {_exp} (canonical={_canon})."
    )
print(f"size = {BUNDLE_ZIP.stat().st_size / (1024**2):.1f} MiB", flush=True)
print("run dir =", BUNDLE_ZIP.parent, flush=True)
print("files:", sorted(p.name for p in BUNDLE_ZIP.parent.iterdir() if p.is_file()), flush=True)

+ /Users/nguyenkz/Documents/code/CS2309.CH201/.venv/bin/python -u /Users/nguyenkz/Documents/code/CS2309.CH201/scripts/run_precision_disk_vram_eval.py --configs fp32,fp16 --n-images 50 --edits-per-image 3 --jobs-manifest /Users/nguyenkz/Documents/code/CS2309.CH201/data/jobs_june17.json --weights-fp32 /Users/nguyenkz/Documents/code/CS2309.CH201/SwiftEdit/swiftedit_weights --weights-fp16 /Users/nguyenkz/Documents/code/CS2309.CH201/SwiftEdit/swiftedit_weights_fp16 --seed 250101049 --max-jobs 150
device=mps configs=['baseline_fp32', 'improved_fp16_cache'] (sequential only, OMP/MKL threads=1, VRAM~16GB safe)
out=/Users/nguyenkz/Documents/code/CS2309.CH201/experimental_data/precision_run_20260724-040029_baseline_fp32+improved_fp16_cache
Jobs từ /Users/nguyenkz/Documents/code/CS2309.CH201/data/jobs_june17.json (150 jobs)
eval_seed(base)=250101049 — per-job hash(base, job_id)
PSNR reference in-run: baseline_fp32
jobs_hash=622489df40d10991

=== baseline_fp32 weights=swiftedit_weights dtype=fp32 

### ⑤ Copy bundle + xem tóm tắt nhanh

Giống notebook gốc: copy zip có tên rõ; in vài dòng `report.md` / `quality_summary.csv` nếu có.

In [6]:
import shutil

from precision_catalog import resolve_config_names

_canon = "+".join(resolve_config_names(EVAL_CONFIGS))
_exp = PROJECT_ROOT / "experimental_data"
_all = sorted(_exp.glob("precision_run_*/bundle.zip"), key=lambda p: p.stat().st_mtime)
_matched = [p for p in _all if _canon in p.parent.name]
BUNDLE_ZIP = (_matched or _all)[-1] if (_matched or _all) else None
assert BUNDLE_ZIP is not None and BUNDLE_ZIP.is_file(), (
    f"Không thấy bundle.zip cho {_canon} trong {_exp}"
)

stamp = BUNDLE_ZIP.parent.name.replace("precision_run_", "")
dl_name = f"swiftedit_bundle_macos_mps_{_canon}_{stamp}.zip"
dl_path = _exp / dl_name
shutil.copy2(BUNDLE_ZIP, dl_path)
print(f"source: {BUNDLE_ZIP}", flush=True)
print(f"copy:   {dl_path} ({dl_path.stat().st_size / (1024**2):.1f} MiB)", flush=True)

run_dir = BUNDLE_ZIP.parent
for name in ("report.md", "quality_summary.csv", "memory.csv", "run_meta.json"):
    p = run_dir / name
    if not p.is_file():
        print(f"(missing) {name}", flush=True)
        continue
    print(f"\n===== {name} =====", flush=True)
    text = p.read_text(encoding="utf-8")
    # Báo cáo ngắn — không dump cả quality.csv
    lines = text.strip().splitlines()
    preview = "\n".join(lines[:40])
    print(preview, flush=True)
    if len(lines) > 40:
        print(f"... ({len(lines) - 40} dòng nữa)", flush=True)

print(
    "\nSo sánh thêm (nếu có nhiều bundle):",
    f"{PY} scripts/compare_precision_runs.py",
    flush=True,
)

source: /Users/nguyenkz/Documents/code/CS2309.CH201/experimental_data/precision_run_20260724-040029_baseline_fp32+improved_fp16_cache/bundle.zip
copy:   /Users/nguyenkz/Documents/code/CS2309.CH201/experimental_data/swiftedit_bundle_macos_mps_baseline_fp32+improved_fp16_cache_20260724-040029_baseline_fp32+improved_fp16_cache.zip (116.0 MiB)

===== report.md =====
# Precision run

- UTC: `2026-07-24T06:31:51.247450+00:00`
- Device: `mps` | torch `2.12.0` | git `a5f52b3`
- Configs: `baseline_fp32`, `improved_fp16_cache`
- Jobs: **150** | `jobs_hash=622489df40d10991`
- Eval seed (base): `250101049` — per-job `hash(base, job_id)`
- PSNR in-run vs: `baseline_fp32`

## Disk

| Label | GiB | Exists |
|---|---:|:---:|
| swiftedit_weights_fp32 | 9.7886 | True |
| swiftedit_weights_fp16 | 4.9421 | True |

## Summary

| Config | n | s/edit | hit | miss | PSNR↔fp32 | peak MB |
|---|---:|---:|---:|---:|---:|---:|
| baseline_fp32 | 150 | 52.135 | None | None | 99.0 | 12366.27 |
| improved_fp16_cache 

### Gợi ý đọc kết quả (đóng góp trên Mac)

Trong `report.md` / CSV của run vừa xong, so `baseline_fp32` vs `improved_fp16_cache`:

1. **s/edit** — FP16+cache nhanh hơn bao nhiêu trên MPS?
2. **cache hit** — edit 2–3/ảnh có hit không?
3. **peak memory (MPS)** — có giảm so với FP32 không?
4. **PSNR vs FP32** — chất lượng còn gần baseline không?

Smoke nhanh: đặt `MAX_JOBS = 6` (2 ảnh × 3) rồi chạy lại cell ⓪ + ④.